<details>
   <summary><b>Table of Contents</b></summary>

   - [1. Review Data](#1-review-data)
   - [2. Train - Test Split](#2-train---test-split)
   - [3. Model Selection](#3-model-selection)
   - [4. Test with Best Model](#4-test-with-best-model)
   - [5. Tuning Model](#5-tuning-model)
   - [6. Evaluate Model](#6-evaluate-model)
     - [6.1. Train - Test Split](#61-train---test-split)
     - [6.2. Cross Validation (K-Fold)](#62-cross-validation-k-fold)
     - [6.3. Visualization](#63-visualization)
   - [7. Save Model](#7-save-model)
</details>

In [ ]:
import warnings
import pandas as pd

from sklearn.pipeline      import Pipeline
from sklearn.compose       import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.svm          import SVR
from sklearn.tree         import DecisionTreeRegressor
from sklearn.neighbors    import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble     import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost              import XGBRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

pd.set_option('display.float_format', lambda x: f'{x:.3e}')

## **1. Review Data**
---

In [ ]:
df = pd.read_csv('../data/data_model.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## **2. Train - Test Split**
---

In [ ]:
X = df.drop('Average Salary', axis=1)
y = df['Average Salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test  shape: {X_test.shape}")
print(f"y_test  shape: {y_test.shape}")

## **3. Model Selection**
---

In [ ]:
def create_regression_pipeline(model_name):
    regression_models = {
        'SVR'                      : SVR(),
        'XGBoost'                  : XGBRegressor(random_state=42),
        'KNeighbors'               : KNeighborsRegressor(),
        'DecisionTree'             : DecisionTreeRegressor(random_state=42),
        'RandomForest'             : RandomForestRegressor(random_state=42),
        'LinearRegression'         : LinearRegression(),
        'HistogramGradientBoosting': HistGradientBoostingRegressor(random_state=42),
    }

    categorical_features = X.select_dtypes(include=['object']).columns.tolist()
    numerical_features   = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
        ]
    )

    regression_pipeline = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('regressor'   , regression_models[model_name])
        ]
    )

    pipeline_dict = {model_name: regression_pipeline}
    return pipeline_dict


def create_all_regression_pipelines():
    regression_model_names = ['RandomForest', 'SVR', 'LinearRegression', 'KNeighbors', 'DecisionTree', 'XGBoost', 'HistogramGradientBoosting']

    all_pipelines = {}
    for model_name in regression_model_names:
        pipeline_dict = create_regression_pipeline(model_name)
        all_pipelines.update(pipeline_dict)
        
    return all_pipelines


def train_evaluate_model_with_df(model_name, model, X_train, y_train, X_test, y_test, results_df):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    model_results = pd.DataFrame({'Model': [model_name], 'MSE': mse, 'MAE': mae, 'R2': r2})
    results_df    = pd.concat([results_df, model_results], ignore_index=True)
    return results_df


def run_pipelines_with_metrics_to_dataframe(all_pipelines, X_train, y_train, X_test, y_test):
    results_df = pd.DataFrame(columns=['Model', 'MSE', 'MAE', 'R2'])

    for model_name, pipeline in all_pipelines.items():
        results_df = train_evaluate_model_with_df(
            model_name, pipeline, 
            X_train, y_train, X_test, y_test, 
            results_df
        )

    results_df = results_df.sort_values(by='R2', ascending=False).reset_index(drop=True)
    return results_df

In [ ]:
all_pipelines = create_all_regression_pipelines()
results_df    = run_pipelines_with_metrics_to_dataframe(all_pipelines, X_train, y_train, X_test, y_test)
results_df

> **Observations**:  
>  
> Best-performing models:  
>  
> - **XGBoost** has the lowest MSE (2.896e+08) and MAE (1.056e+04) with the highest R² (0.8153), making it the top performer.  
> - **Random Forest** follows closely with similar MSE and MAE but slightly lower R² (0.8127).  
>  
> Moderate performance:  
>  
> - **Histogram Gradient Boosting**, **Decision Tree**, and **Linear Regression** show decent results but with higher errors and lower R² scores (~0.70).  
>  
> Poor performance:  
>  
> - **K-Neighbors** has significantly higher MSE (8.030e+08) and MAE (2.114e+04), leading to a poor R² (0.4878).  
> - **SVR** performs worst, with extremely high MSE (1.616e+09) and MAE (3.117e+04), and a negative R² (-0.0378), indicating it fails to explain variance.  

> **Conclusion**:  
>  
> - **XGBoost** is the best model, offering the best balance of error minimization and predictive power.  
> - **Random Forest** is a strong alternative, with only a slight drop in performance.  
> - **SVR** and **K-Neighbors** are ineffective for this dataset and should be avoided.  

## **4. Test with Best Model**
---

In [ ]:
xgb_res        = XGBRegressor()
default_params = xgb_res.get_params()

desired_params          = ['reg_lambda', 'reg_alpha', 'n_estimators', 'max_depth', 'learning_rate', 'gamma', 'colsample_bytree']
selected_default_params = {param: default_params[param] for param in desired_params}

default_params_df = pd.DataFrame([selected_default_params])
default_params_df = default_params_df.transpose()
default_params_df.columns = ['Value']

print('XGBRegressor default params:')
default_params_df.T

In [ ]:
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features   = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

xgb_res = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', XGBRegressor())
    ]
)

xgb_res.fit(X_train, y_train)
y_pred = xgb_res.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"MSE: {mse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R2 : {r2:.3f}")

## **5. Implementing Optuna for Hyperparameter Tuning**
---

In [ ]:
import optuna
from optuna.pruners          import MedianPruner
from sklearn.base            import clone
from sklearn.model_selection import KFold, train_test_split
from sklearn.compose         import ColumnTransformer
from sklearn.preprocessing   import OneHotEncoder, StandardScaler
from xgboost                 import XGBRegressor
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Read data
df = pd.read_csv('../data/data_model.csv')
X  = df.drop('Average Salary', axis=1)
y  = df['Average Salary']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features   = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ]
)

In [ ]:
def objective(trial: optuna.Trial):
    """Define the objective function for Optuna optimization."""

    # Hyperparameter space
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 1500, step=50),
        'learning_rate'   : trial.suggest_float('learning_rate', 1e-3, 1, log=True),
        'max_depth'       : trial.suggest_int('max_depth', 3, 21),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
        'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma'           : trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        # For faster training
        'tree_method' : 'hist',
        'random_state': 42,
        'n_jobs'      : -1,
    }

    # KFold cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []

    # Cross-validation loop and early stopping (each fold will have its own eval_set)
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), start=1):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        
        prep      = clone(preprocessor)
        X_tr_enc  = prep.fit_transform(X_tr)
        X_val_enc = prep.transform(X_val)

        reg = XGBRegressor(**params, early_stopping_rounds=50)
        reg.fit(
            X_tr_enc, y_tr,
            eval_set = [(X_val_enc, y_val)],
            verbose  = False
        )

        y_pred = reg.predict(X_val_enc)
        rmse   = np.sqrt(mean_squared_error(y_val, y_pred))
        rmse_scores.append(rmse)

        # Report intermediate results to Optuna and handle pruning
        trial.report(rmse, step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # Object: minimize mean RMSE
    return float(np.mean(rmse_scores))

In [ ]:
# Create Optuna study
study = optuna.create_study(
    direction = 'minimize',
    sampler   = optuna.samplers.TPESampler(seed=42),
    pruner    = MedianPruner(n_warmup_steps=2),
)
study.optimize(objective, n_trials=2000, show_progress_bar=True, n_jobs=-1)

In [ ]:
# Report the best trial
print('Best trial :', study.best_trial.number)
print('Best RMSE  :', study.best_value)
print('Best params:')
for k, v in study.best_params.items():
    print(f"\t{k}: {v}")

In [ ]:
# Re-train the model with the best hyperparameters
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

prep      = clone(preprocessor)
X_tr_enc  = prep.fit_transform(X_tr)
X_val_enc = prep.transform(X_val)

best_reg = XGBRegressor(
    **study.best_params,
    tree_method           = 'hist',
    random_state          = 42,
    n_jobs                = -1,
    early_stopping_rounds = 50,
)

best_reg.fit(
    X_tr_enc, y_tr,
    eval_set = [(X_val_enc, y_val)],
    verbose  = False,
)

## **6. Evaluation**
---

In [ ]:
from sklearn.pipeline import Pipeline
best_model = Pipeline([
    ('preprocessor', prep),
    ('regressor'   , best_reg),
])

y_pred = best_model.predict(X_test)
mse    = mean_squared_error(y_test, y_pred)
rmse   = np.sqrt(mse)
mae    = mean_absolute_error(y_test, y_pred)
r2     = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.3f}")
print(f"MAE : {mae:.3f}")
print(f"R2  : {r2:.3f}")

## **7. Save Model**
---

In [ ]:
import joblib

joblib.dump(best_reg, '../model/xgb_optuna_tuning.pkl')
print('XGBoost model with Optuna tuning saved to ../model/xgb_optuna_tuning.pkl')